# Migration: Dept_pkg
**Source File:** Dept_pkg.sql  
**Target:** Databricks Spark SQL / Delta Lake

In [ ]:
dbutils.widgets.text("V_Dept", "")
dbutils.widgets.text("ODI_SESS_NO", "75492154-c22b-4f4d-85f5-5f7be13771ae")
dbutils.widgets.text("ETL_PROC_WID", "")

### ETL Extraction Parameter Setup

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Get ETL Run Date
CREATE OR REPLACE TEMPORARY VIEW v_last_run_dt AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_trg.log_table1 
         WHERE pkg_name = 'Dept_pkg'),
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS last_run_dt;

In [ ]:
%sql
-- SCEN_TASK_NO {2}: Update Log Table
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

### Staging Table (C$)

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop Staging Table
DROP TABLE IF EXISTS workspace.odi_trg.c_filter_stg;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create Staging Table
CREATE TABLE workspace.odi_trg.c_filter_stg
(
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Insert into Staging
INSERT INTO workspace.odi_trg.c_filter_stg
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    SRC.DEPARTMENT_ID,
    SRC.DEPARTMENT_NAME,
    SRC.MANAGER_ID,
    SRC.LOCATION_ID,
    SRC.LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC
WHERE SRC.LAST_UPD_DT >= to_date('${V_Dept}', 'dd-MM-yy');

In [ ]:
%sql
-- SCEN_TASK_NO {60}: Optimize Staging Stats
OPTIMIZE workspace.odi_trg.c_filter_stg;

### Flow Table (I$)

In [ ]:
%sql
-- SCEN_TASK_NO {80}: Drop Flow Table
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {90}: Create Flow Table
CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
    ODI_ROW_ID      STRING, 
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    IND_UPDATE      STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {100}: Insert into Flow (New or Changed Records)
INSERT INTO workspace.odi_trg.i_trg_departments_flow
SELECT 
    CAST(monotonically_increasing_id() AS STRING) AS ODI_ROW_ID,
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT,
    'I' AS IND_UPDATE
FROM workspace.odi_trg.c_filter_stg AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.odi_trg.trg_departments AS T
    WHERE T.DEPARTMENT_ID = S.DEPARTMENT_ID 
      AND (COALESCE(T.DEPARTMENT_NAME, '##') = COALESCE(S.DEPARTMENT_NAME, '##'))
      AND (COALESCE(T.MANAGER_ID, -1) = COALESCE(S.MANAGER_ID, -1))
      AND (COALESCE(T.LOCATION_ID, -1) = COALESCE(S.LOCATION_ID, -1))
      AND (COALESCE(T.LAST_UPD_DT, to_timestamp('1900-01-01', 'yyyy-MM-dd')) = COALESCE(S.LAST_UPD_DT, to_timestamp('1900-01-01', 'yyyy-MM-dd')))
);

In [ ]:
%sql
-- SCEN_TASK_NO {110, 120}: Optimize Flow Table
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_trg_departments_flow ZORDER BY (DEPARTMENT_ID);

### Error Control and PK Validation

In [ ]:
%sql
-- SCEN_TASK_NO {130, 140}: Audit Initialization
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME    STRING,
    SCHEMA_NAME     STRING,
    RESOURCE_NAME   STRING,
    FULL_RES_NAME   STRING,
    ERR_TYPE        STRING,
    ERR_MESS        STRING,
    CHECK_DATE      TIMESTAMP,
    ORIGIN          STRING,
    CONS_NAME       STRING,
    CONS_TYPE       STRING,
    ERR_COUNT       BIGINT
) USING DELTA;

DELETE FROM workspace.odi_trg.snp_check_tab
WHERE SCHEMA_NAME = 'odi_trg'
  AND ORIGIN = '(7)test.Dept_MAp'
  AND ERR_TYPE = 'F';

In [ ]:
%sql
-- SCEN_TASK_NO {150, 160}: Error Table Initialization
CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_departments
(
    ODI_ROW_ID      STRING,
    ODI_ERR_TYPE    STRING, 
    ODI_ERR_MESS    STRING,
    ODI_CHECK_DATE  TIMESTAMP, 
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    ODI_ORIGIN      STRING,
    ODI_CONS_NAME   STRING,
    ODI_CONS_TYPE   STRING,
    ODI_PK          STRING,
    ODI_SESS_NO     STRING
) USING DELTA;

DELETE FROM workspace.odi_trg.e_trg_departments
WHERE (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(7)test.Dept_MAp');

In [ ]:
%sql
-- SCEN_TASK_NO {180}: PK Violation Check
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_ORIGIN,
    ODI_CHECK_DATE,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    uuid(),
    '${ODI_SESS_NO}', 
    ODI_ROW_ID,
    'F', 
    'ODI-15064: The primary key PK_department_id is not unique.',
    '(7)test.Dept_MAp',
    current_timestamp(),
    'PK_department_id',
    'PK',	
    F.DEPARTMENT_ID,
    F.DEPARTMENT_NAME,
    F.MANAGER_ID,
    F.LOCATION_ID,
    F.LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow AS F
WHERE EXISTS (
    SELECT 1
    FROM workspace.odi_trg.i_trg_departments_flow AS SUB
    WHERE SUB.DEPARTMENT_ID = F.DEPARTMENT_ID
    GROUP BY SUB.DEPARTMENT_ID
    HAVING COUNT(1) > 1
);

In [ ]:
%sql
-- SCEN_TASK_NO {190}: Not Null Check (DEPARTMENT_NAME)
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_CHECK_DATE,
    ODI_ORIGIN,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    uuid(),
    '${ODI_SESS_NO}', 
    ODI_ROW_ID,
    'F', 
    'ODI-15066: The column DEPARTMENT_NAME cannot be null.',
    current_timestamp(),
    '(7)test.Dept_MAp',
    'DEPARTMENT_NAME',
    'NN',	
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow
WHERE DEPARTMENT_NAME IS NULL;

In [ ]:
%sql
-- SCEN_TASK_NO {210}: Remove Errors from Flow
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING (
    SELECT DISTINCT ODI_ROW_ID 
    FROM workspace.odi_trg.e_trg_departments
    WHERE ODI_SESS_NO = '${ODI_SESS_NO}'
) AS E
ON T.ODI_ROW_ID = E.ODI_ROW_ID
WHEN MATCHED THEN DELETE;

In [ ]:
%sql
-- SCEN_TASK_NO {220}: Audit Summary
INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME,
    RESOURCE_NAME,
    FULL_RES_NAME,
    ERR_TYPE,
    ERR_MESS,
    CHECK_DATE,
    ORIGIN,
    CONS_NAME,
    CONS_TYPE,
    ERR_COUNT
)
SELECT 	
    'odi_trg',
    'TRG_DEPARTMENTS',
    'workspace.odi_trg.trg_departments',
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE,
    COUNT(1) 
FROM workspace.odi_trg.e_trg_departments AS E
WHERE E.ODI_ERR_TYPE = 'F'
  AND E.ODI_ORIGIN = '(7)test.Dept_MAp'
GROUP BY 
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE;

### Target Integration (MERGE)

In [ ]:
%sql
-- SCEN_TASK_NO {230}: Flag Records for Update
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING (
    SELECT DEPARTMENT_ID
    FROM workspace.odi_trg.trg_departments
) AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

In [ ]:
%sql
-- SCEN_TASK_NO {270, 280}: Merge into Target
MERGE INTO workspace.odi_trg.trg_departments AS T
USING workspace.odi_trg.i_trg_departments_flow AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.DEPARTMENT_NAME = S.DEPARTMENT_NAME,
    T.MANAGER_ID      = S.MANAGER_ID,
    T.LOCATION_ID     = S.LOCATION_ID,
    T.LAST_UPD_DT     = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
VALUES
(
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT
);

### Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO {340, 360}: Drop Temporary Tables
DROP TABLE IF EXISTS workspace.odi_trg.c_filter_stg;
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;